# Restaurant AI - People Detection

This notebook demonstrates YOLOv8-based person detection for restaurant analytics.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
from pathlib import Path
import time

## 2. Initialize Detector

In [ ]:
# Initialize YOLOv8 person detector
detector = PersonDetector(model_path='yolov8n.pt', confidence=0.5)
print("Person detector initialized with YOLOv8n model")

## 3. Load Video

In [ ]:
# Load video file (use sample video or camera)
VIDEO_PATH = '../data/sample_video.mp4'  # Replace with your video path

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    print("Video not found. Using camera instead.")
    cap = cv2.VideoCapture(0)  # Use camera

video_info = get_video_info(cap)
print(f"Video Info: {video_info}")

## 4. Detection on Sample Frames

In [ ]:
# Process video and display detections
frame_count = 0
max_frames = 30  # Process 30 frames for demo

while frame_count < max_frames:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Detect persons
    boxes, confidences, class_ids = detector.detect(frame)
    
    # Draw results
    output = draw_boxes(frame, boxes, ids=None, confidences=confidences)
    
    # Display
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
    plt.title(f'Frame {frame_count} - {len(boxes)} persons detected')
    plt.axis('off')
    plt.show()
    
    frame_count += 1
    if frame_count >= 5:  # Show first 5 frames
        break

cap.release()

## 5. Real-time Detection with Camera

In [ ]:
# Real-time detection from camera
def run_realtime_detection(camera_id=0, confidence=0.5):
    """Run real-time person detection from camera."""
    cap = cv2.VideoCapture(camera_id)
    detector = PersonDetector(confidence=confidence)
    
    print("Press 'q' to quit")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        boxes, confidences, class_ids = detector.detect(frame)
        output = draw_boxes(frame, boxes, confidences=confidences)
        
        cv2.putText(output, f'Persons: {len(boxes)}', (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        cv2.imshow('Person Detection', output)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()

# Uncomment to run:
# run_realtime_detection()

## 6. Batch Processing

In [ ]:
# Batch processing example
cap = cv2.VideoCapture(VIDEO_PATH)
frames = []

# Read 10 frames
for _ in range(10):
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(frame)

cap.release()

# Process batch
detections = detector.detect_batch(frames)

# Show results
for i, (boxes, confidences, class_ids) in enumerate(detections):
    print(f"Frame {i}: {len(boxes)} persons detected")

## Summary

This notebook covers:
- Loading YOLOv8 model for person detection
- Processing video files or camera streams
- Displaying bounding boxes with confidence scores
- Real-time detection mode
- Batch processing for efficiency